# Qwen3.5-4B: K2 reasons + reward-filtered fine-tune of an old checkpoint → new public submission

Built by `student/make_rl_notebook.py`. One session, **GPU T4 x2, Internet on**, sized to have the new model ready by
`DEADLINE_VN` (default 20:30: the eval notebook's 2 h and the Docker build's 1 h still follow before 24:00).

**What it does**
1. **Rollouts** — the old checkpoint acts on train rows in the real `ToolEnv` (ASR text, greedy), 2 GPUs.
   Reward = the official score against train gold. Row order: 66 gate rows (never trained on here), the other 262
   held-out rows, then train rows until the stage's time is up.
2. **K2 Horizon** (background, CPU, optional) — `solve.py --backend ifm --why` on the same train rows: one tool per turn
   with one written reason. Used only where K2 itself scored 1.0 in gold order (`why_from_traces.py`).
3. **Policy improvement** — a fresh LoRA on the old weights, trained on the rows selected by reward: every failed rollout
   twice (gold trajectory as the target, K2's reasons where available), then successes on rows the checkpoint never
   trained on, then other successes, as many as fit in the stage's time. Merged → `llm_model/`.
4. **Gate** — the new model on the 66 gate rows against the old model's rollouts there (~12 min).
5. **Public run** only with `PUBLIC_RUN = True` (about 2 h more). By default the eval notebook makes the submission:
   attach this notebook's output (`llm_model/`, `lifemate.py`) to it.

**Eval notebook**: its held-out score is **not valid for this model** — 262 of the 328 held-out rows are trained on
here. Only the gate line compares old and new fairly. Run the eval notebook with `EVAL = False` (public only).

This is one round of reward-filtered imitation (rollout → reward → train on the corrected failures), not policy-gradient
RL: sampling several trajectories per row and several rounds do not fit in the hours left.

**Attach (Add Input)**
1. the competition data package (private),
2. the old checkpoint's notebook output: `llm_model/` + `lifemate.py` (exactly one `llm_model` among the inputs),
3. the 01c output: `asr_transcripts.json`, `asr_base_train.json` (or `asr_xfit_train.json`), `vi_itn.py`,
4. `rationales.json` — the file the checkpoint was trained with (all 1,545 train ids),
5. optional, for K2: the `zola_deepseek_bundle` dataset and the secret `IFM_API_KEY` (Add-ons → Secrets, attached).
   Without them the run goes on with the reasons of `rationales.json`.

**Run**: optional `SMOKE = True` first (~20 min: 8 rows per GPU, 3 updates, merge). Then
`SMOKE = False` → *Save Version → Save & Run All (Commit)*. The first code cell prints the time plan; it stops at once
if too little time is left.

**Output**: `llm_model/` + `lifemate.py` (new weights), `rl_summary.json` (gate result), `pred_*.jsonl`
(rollouts with rewards), `rl_rows.json`, `rationales_rl.json`, `traces_train_k2_why.jsonl`, `train.log`.

**Rules**: K2 sees train rows only and only to make training data (R-RES-2/3); nothing from public_test is trained on;
the public run uses the local model and 01c text only. Never attach `public_test_rationales.json` or public traces.

In [ ]:
# 1. GPUs and packages (the versions the checkpoint was trained with)
import torch
assert torch.cuda.device_count() == 2, "Set Settings > Accelerator to GPU T4 x2"
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(2)])
!pip install -q "transformers==5.22.1" "peft==0.21.2" "accelerate==1.15.0"
!pip uninstall -q -y torchao  # unused, and peft refuses the old version Kaggle ships
!python -c "import peft, transformers; import transformers.models.qwen3_5.modeling_qwen3_5 as q; assert hasattr(q, 'torch_chunk_gated_delta_rule'); print('transformers', transformers.__version__, 'peft', peft.__version__)"
assert _exit_code == 0, "the training script needs transformers 5.22.1 with the Qwen3.5 model code: is Internet on?"

In [ ]:
# 2. Settings
import datetime as dt, glob, json, os, random, shutil, statistics as st, subprocess, sys, time
from pathlib import Path

SMOKE = False                      # True: 8 rows per GPU, 3 updates, 16 public samples (~25 min). Then False and Save & Run All
DEADLINE_VN = "2026-11-16 20:30"   # Vietnam time: llm_model/ is ready by then. After it: the commit saves ~9 GB of output, the eval notebook (2 h), the Docker build (1 h), all before 24:00
PUBLIC_RUN = False                 # False: the eval notebook makes the submission from llm_model/. True: also run public_test here (about 2 h more, before the deadline)
ROLL_CAP, TRAIN_CAP = 55, 80       # minutes at most for the rollouts and for training; both shrink when less time is left
PUBLIC_MIN = 120                   # first estimate for the public run, replaced by the speed the rollouts measure
SEC_PER_CONV = 7.7                 # training seconds per conversation on 2x T4 (bowen: 459 updates of 8 in 7.8 h)
USE_K2, K2_WORKERS, K2_ROWS = True, 4, 600  # K2 Horizon reasons for train rows; 10M tokens per key per day
os.environ["SMOKE"] = "1" if SMOKE else "0"
DEADLINE = dt.datetime.strptime(DEADLINE_VN, "%Y-%m-%d %H:%M").replace(tzinfo=dt.timezone(dt.timedelta(hours=7))).timestamp()


def left():
    """Minutes to the deadline."""
    return (DEADLINE - time.time()) / 60


def run_logged(cmd, log, env=None):
    with open(log, "a", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", env=env)
        for line in p.stdout:
            print(line, end="")
            f.write(line)
            f.flush()
        return p.wait()


GATE_MIN = 15  # the new model on the 66 gate rows
flex = left() - 10 - 15 - GATE_MIN - PUBLIC_MIN * PUBLIC_RUN  # what rollouts + training can share: minus model loads, merge, gate, public run
print(f"{left():.0f} min to {DEADLINE_VN} (+07); rollouts and training share {flex:.0f} min")
assert SMOKE or flex >= 45, "too late for rollouts + training before the deadline: move DEADLINE_VN at your own risk"
ROLL_MIN = 6 if SMOKE else min(ROLL_CAP, 0.4 * flex)

In [ ]:
%%writefile settings.py
"""Shared by infer.py (neweval's settings), train_llm.py (bowen_sft's settings) and merge.py. The notebook passes the
per-stage values (model to load, training hours) as environment variables."""
import glob
import json
import os
from pathlib import Path

IN = os.environ.get("ZOLA_IN", "/kaggle/input")
SMOKE = os.environ.get("SMOKE") == "1"
SMOKE_STEPS = 3
SEED = 42

# inference, as neweval
DTYPE = "float16"
SHARDS = 2  # one process per T4
MAX_STEPS = 8
MAX_NEW_TOKENS = 320
TIME_BUDGET = 60.0  # per sample at most (the slowest sample of neweval's T4 run took 56.7 s); infer.py lowers it when the deadline is near
STRIP_MARKS = True
ASR_JSON = "asr_all.json"  # public_test: 01c text; train: the noisy text the checkpoint was trained on

# training, as bowen_sft except EPOCHS, LR, WARMUP_STEPS, MAX_HOURS, SAVE_STEPS and the base model
FINAL = False  # unused: the rows come from rl_rows.json
FP16 = True
MAX_LEN = 9000
DELTA_PART = 512
EPOCHS = 1  # rl_rows.json already repeats the failed rows
GRAD_ACCUM = 8
LR = 1e-4  # half of the first fine-tune: a short correction on already trained weights
WARMUP_STEPS = 5
LORA_R = 32
NOISE_RATE = 0.5
HISTORY_CALLS = True
WHY = json.load(open("rl_config.json"))["why"] if os.path.exists("rl_config.json") else True  # does the checkpoint write a reason first? Measured on the rollouts
WHY_HISTORY = False
MASK_RATE = 0.3
MAX_HOURS = float(os.environ.get("TRAIN_HOURS", 1.0))  # train_llm.py stops, saves and exits cleanly at this point
MIN_FREE_GB = 3
SAVE_STEPS = 100000  # no checkpoints on the way: the run is short and stops by time
RESUME, CHECKPOINT = False, None

ROOT = Path(sorted(glob.glob(f"{IN}/**/train/train.jsonl", recursive=True))[0]).parents[1]
_llm = sorted(glob.glob(f"{IN}/**/llm_model/config.json", recursive=True))
assert len(_llm) == 1, f"attach exactly one checkpoint output with llm_model/, found {_llm}"
LLM_OLD = str(Path(_llm[0]).parent)
LLM = os.environ.get("LLM_DIR") or LLM_OLD  # the model infer.py loads
MODEL_ID = LLM_OLD  # training starts from the old checkpoint's merged weights
LIFEMATE_DIR = "."  # the checkpoint's own lifemate.py, copied here by the notebook: the prompt must match the weights
_noisy = glob.glob(f"{IN}/**/asr_xfit_train.json", recursive=True) or glob.glob(f"{IN}/**/asr_base_train.json", recursive=True)
assert _noisy, "attach the 01c output: asr_base_train.json (or asr_xfit_train.json) is the train text"
NOISY_ASR = _noisy[0]
_pub = glob.glob(f"{IN}/**/asr_transcripts.json", recursive=True)
assert len(_pub) == 1, f"attach exactly one 01c output with asr_transcripts.json, found {_pub}"
ASR_PUBLIC = _pub[0]
_why = glob.glob(f"{IN}/**/rationales.json", recursive=True)
assert _why, "attach rationales.json, the file the checkpoint was trained with"
WHY_BASE, WHY_JSON = _why[0], "rationales_rl.json"

In [ ]:
%%writefile infer.py
"""One shard of the agent: python infer.py <split> <gold|asr> <shard>. One process per split, each split has its own zaic_tools."""
import json
import os
import re
import sys
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from transformers.integrations import sdpa_attention

from settings import *

sdpa_attention.use_gqa_in_sdpa = lambda *args, **kwargs: False  # a T4 has no grouped-key kernel: Qwen3.5 fell back to float32 math and ran out of memory on long prompts (public_0110, public_0157)

sys.path[:0] = [LIFEMATE_DIR, str(ROOT / sys.argv[1])]
import lifemate as lm
from zaic_tools import ToolEnv

split, mode, shard = sys.argv[1], sys.argv[2], int(sys.argv[3])
set_seed(SEED)
tok = AutoTokenizer.from_pretrained(LLM)
model = AutoModelForCausalLM.from_pretrained(LLM, dtype=DTYPE, device_map="auto" if torch.cuda.is_available() else None, trust_remote_code=True).eval()
stop_id = tok.convert_tokens_to_ids(lm.STOP)  # not every repo has a generation_config, so the default eos may never fire
catalog = lm.load_catalog(ROOT, split)
rows = lm.load_split(ROOT, split)
asr = json.load(open(ASR_JSON, encoding="utf-8")) if mode == "asr" else {}
if split == "train":  # rollout / gate rows in the order of the ids file. Scored with gold, so train rows only
    _by = {x["id"]: x for x in rows}
    rows = [_by[i] for i in json.load(open(os.environ["IDS_JSON"]))]
rows = rows[shard::SHARDS][: 8 if SMOKE else None]


@torch.inference_mode()
def generate(msgs):
    ids = tok(lm.prompt_text(msgs), return_tensors="pt", add_special_tokens=False).to(model.device)
    assert ids.input_ids.shape[1] + MAX_NEW_TOKENS <= 16384, "over the competition context limit"
    out = model.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, eos_token_id=stop_id, pad_token_id=stop_id)
    return tok.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True)


def clean(text):
    return re.sub(r"[\[\]]", "", text) if STRIP_MARKS else text  # norm() removes all punctuation, so every bracket here is a mark


def missing(name, args, installed):
    """The ask gold would make for an installed tool that misses a required or one-of parameter, else None. Reads too: gold asks for them 7 times (flights.search date). A one-of group names its first member, as gold does for ewallet.transfer (contact_id)."""
    tool = catalog.get(name)
    if tool is None or name.split(".")[0] not in installed:
        return None
    p = tool.get("parameters", {})
    empty = lambda k: args.get(k) in (None, "", [], {})  # zaic_tools counts these as missing
    need = [k for k in p.get("required", []) if empty(k)]
    if p.get("x-one-of") and all(any(empty(k) for k in grp) for grp in p["x-one-of"]):
        need.append(p["x-one-of"][0][0])
    return {"tool_name": name, "params": need} if need else None


def run(item):
    env = ToolEnv(split, item["id"], root=ROOT)  # exactly one per sample, every call on it goes into the submission
    texts = lm.gold_texts(item) if mode == "gold" else {c: clean(asr[f"{split}/{c}"]) for c in lm.clips(item)}
    msgs = lm.build_messages(item, texts, catalog)
    t0, invalid, failed, stop, raw = time.time(), 0, 0, "max_steps", []
    retried = False
    try:
        for _ in range(MAX_STEPS):
            if time.time() - t0 > budget:
                stop = "budget"
                break
            raw.append(generate(msgs))
            action = lm.parse_action(raw[-1])
            if action is None:  # parse_action returns None for both, and it stays as saved with the weights
                stop = "done" if '"done"' in raw[-1] else "unparsable"
                break
            call = {"name": action[0], "arguments": action[1]}
            said = json.JSONDecoder().raw_decode(raw[-1][raw[-1].index("{"):])[0]  # the model's own JSON, so a trained "why" stays in context. Equals call for other models
            check = env.validate(*action)
            if not check["ok"]:  # validate is not logged: a call we never make costs no U penalty
                ask = missing(*action, item["context"]["installed_apps"])
                if ask and env.validate(lm.ASK, ask)["ok"]:  # gold ends on its ask too
                    env.call(lm.ASK, ask)
                    stop = "repaired"
                    break
                if not retried:  # one more try with the error as the result (4 of 5 invalid stops on public called an uninstalled contacts.search)
                    retried = True
                    msgs += [lm.asst(said), lm.result_msg(check)]
                    continue
                invalid, stop = 1, "invalid"
                break
            if env.get_tool_calls().count(call) >= 2:  # greedy loop, gold repeats a call at most once
                stop = "repeat"
                break
            if catalog.get(call["name"], {}).get("kind") == "read" and [c["name"] for c in env.get_tool_calls()].count(call["name"]) >= 2:  # gold reads one tool 3+ times in 1 of 1,545 samples; LFM3 ver3 retried empty searches 127 times on public, then sometimes wrote on a guess
                stop = "repeat"
                break
            result = env.call(*action)
            msgs += [lm.asst(said), lm.result_msg(result)]
            if "query" in action[1] and not catalog[call["name"]]["parameters"].get("required") and any(v == [] for v in result.values()):  # gold never reads an empty list (0 of 234 contacts.search): the query was misheard, so list everything once and let the model pick. 25 public samples hit this (contacts, event_tickets); a full list fits MAX_RESULT_CHARS
                msgs += [lm.asst({"name": call["name"], "arguments": {}}), lm.result_msg(env.call(call["name"], {}))]
            if call["name"] == lm.ASK:  # gold always ends here
                stop = "ask"
                break
    except Exception as e:  # keep the calls already made, a crash must not lose the whole line
        failed, stop = 1, "crash"
        print(item["id"], repr(e), flush=True)
    return {"id": item["id"], "tool_calls": env.get_tool_calls(), "state_hash": env.get_state_hash(), "sec": time.time() - t0, "invalid": invalid, "failed": failed, "stop": stop, "raw": raw}


STOP_AT, budget = float(os.environ["STOP_AT"]), TIME_BUDGET
with open(f"pred_{os.environ['TAG']}_{split}_{shard}.jsonl", "w", encoding="utf-8") as f:
    for n, item in enumerate(rows):
        left = STOP_AT - time.time()
        if split == "train" and left < 0:  # rollouts and the gate: rows not reached are simply not used
            break
        budget = min(TIME_BUDGET, max(5.0, left / (len(rows) - n)))  # public: the time left is shared out, so every sample gets its line before the deadline (R-RUN-3)
        line = run(item)
        if split == "train":  # scoring only: the second env replays gold for its hash, nothing here reaches a submission
            gold = item["label"]["tool_calls"]
            ref = ToolEnv(split, item["id"], root=ROOT)
            for c in gold:
                ref.call(c["name"], c["arguments"])
            line["score"] = lm.score(line["tool_calls"], gold, line["state_hash"], ref.get_state_hash(), catalog)
            line["unseen"] = bool(lm.used_apps(item) & lm.HOLDOUT_APPS)
        f.write(json.dumps(line, ensure_ascii=False) + "\n")
        f.flush()
        if n % 25 == 0:
            print(split, mode, shard, n, "/", len(rows), f"{line['sec']:.1f}s", flush=True)

In [ ]:
%%writefile train_llm.py
"""LoRA fine-tune of the LLM on gold tool trajectories. One process, the model sharded across both T4s: a full copy per GPU runs out of memory."""
import json
import os
import resource
import random
import re
import sys
import time
import unicodedata

import torch
import torch.nn.functional as F
import transformers.integrations.sdpa_attention as sdpa
import transformers.models.qwen3_5.modeling_qwen3_5 as qwen
from peft import LoraConfig, get_peft_model
from torch.utils.checkpoint import checkpoint
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainerCallback, TrainingArguments, set_seed

import lifemate as lm
from settings import *

sys.path.insert(0, os.path.dirname(NOISY_ASR))
from vi_itn import itn  # the ASR output's own ITN, so clean text gets the same digits as transcripts

START = time.time()


def free_gb():  # the whole machine, so it also counts the notebook kernel, not only this process
    return int(next(l for l in open("/proc/meminfo") if l.startswith("MemAvailable")).split()[1]) / 2**20


set_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
rows = lm.load_split(ROOT, "train")
catalog = lm.load_catalog(ROOT, "train")
_by = {x["id"]: x for x in rows}
rows = [_by[i] for i in json.load(open("rl_rows.json"))]  # reward-selected train rows, failed rollouts twice
noisy = {k: re.sub(r"[\[\]]", "", v) for k, v in json.load(open(NOISY_ASR, encoding="utf-8")).items()}  # 03c removes the tagger's marks, so training must too
whys = json.load(open(WHY_JSON, encoding="utf-8")) if WHY else {}
assert all(x["id"] in whys for x in rows) or not WHY, "rationales.json misses samples, re-run gen_rationales.py"
missing = sum(k not in noisy for x in rows for k in lm.gold_texts(x))
assert not missing, f"{missing} training clips have no ASR text, was this output a smoke run of 01?"


def old_style_tone(word):  # asr_common.norm from the 01 notebooks, so clean text takes the transcripts' form
    d = unicodedata.normalize("NFD", word)
    tones = [c for c in d if c in "̣̀́̃̉"]
    if len(tones) != 1:
        return word
    base = unicodedata.normalize("NFC", "".join(c for c in d if c not in tones))
    if base.endswith(("oa", "oe")) or (base.endswith("uy") and not base.endswith("quy")):
        return unicodedata.normalize("NFC", base[:-2] + base[-2] + tones[0] + base[-1])
    return word


def asr_style(s):
    """Test input has no capitals or punctuation, and its numbers come from ITN: clean text goes through the same norm and ITN as transcripts."""
    s = re.sub(r"[^\w\s]|_", " ", unicodedata.normalize("NFC", s).lower())
    return itn(" ".join(old_style_tone(w) for w in s.split()))


def conversation(x):
    texts = lm.gold_texts(x)
    texts = {k: noisy[k] for k in texts} if random.random() < NOISE_RATE else {k: asr_style(v) for k, v in texts.items()}  # real ASR errors, or the clean words in transcript form
    msgs = lm.build_messages(x, texts, catalog, random)
    first = len(msgs)
    msgs += lm.gold_steps(x)
    labelled = lm.targets(msgs, first, HISTORY_CALLS)
    if WHY:  # the reason goes first in the same JSON, so the model writes it before deciding. Reasons cover history calls, final calls, done
        w = whys[x["id"]][-len(labelled):]
        assert len(w) == len(labelled), x["id"]
        for i, why in zip(sorted(labelled), w):
            if WHY_HISTORY or i >= first:  # history calls stay as inference renders them, without a reason
                msgs[i] = lm.asst({"why": why} | json.loads(msgs[i]["content"]))
        assert WHY_HISTORY or not any('"why"' in m["content"] for m in msgs[:first]), x["id"]
    if random.random() < MASK_RATE:
        msgs = lm.mask_apps(msgs, x["context"]["installed_apps"], random)
    return lm.encode(tok, msgs, labelled)


before = len(rows)
rows = [x for x in rows if len(conversation(x)["input_ids"]) <= MAX_LEN]
print(f"{len(rows)} conversations, {before - len(rows)} dropped as longer than {MAX_LEN} tokens, ASR noise: {os.path.basename(NOISY_ASR)}, history reasons: {WHY_HISTORY}, history calls learnt: {HISTORY_CALLS}, reasons: {WHY}, apps masked: {MASK_RATE}")
if SMOKE:  # longest conversations, so an out-of-memory error shows up now and not hours in
    rows = sorted(rows, key=lambda x: -len(conversation(x)["input_ids"]))[: SMOKE_STEPS * GRAD_ACCUM * 2]


class Conversations(torch.utils.data.Dataset):
    def __len__(self):
        return len(rows)

    def __getitem__(self, i):
        return conversation(rows[i])


def first_not_finite(model, inputs, keep):
    """Rerun a failed conversation with a hook on every module: inner modules finish first, so the first hit is where it starts."""
    names, hits = {m: n for n, m in model.named_modules()}, []

    def check(module, args, out):
        out = out[0] if isinstance(out, tuple) else out
        if torch.is_tensor(out) and out.is_floating_point() and not torch.isfinite(out).all():
            hits.append(f"{names[module]} ({out.dtype})")

    hooks = [m.register_forward_hook(check) for m in names]
    with torch.no_grad():
        model(**inputs, logits_to_keep=keep)
    for h in hooks:
        h.remove()
    return hits[0] if hits else "none, the rerun was finite"


class TargetOnlyLoss(Trainer):
    """Run lm_head on target tokens only: Qwen3.5 has 248k logits per token, about 3 GB for a 6k token prompt."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")[0, 1:]
        keep = (labels != -100).nonzero()[:, 0]
        out = model(**inputs, logits_to_keep=keep.tolist())  # a list: accelerate would move a tensor to the first GPU, and the model ends on the second
        loss = F.cross_entropy(out.logits[0].float(), labels[keep].to(out.logits.device))
        if not torch.isfinite(loss):
            bad_weights = sum(not torch.isfinite(p).all() for p in model.parameters() if p.requires_grad)
            scaler = self.accelerator.scaler.get_scale() if self.accelerator.scaler else None
            raise RuntimeError(f"Loss is not finite at step {self.state.global_step}, {inputs['input_ids'].shape[1]} tokens. Trainable weights not finite: {bad_weights}. GradScaler scale: {scaler}. First module not finite: {first_not_finite(model, inputs, keep.tolist())}. If this is fp16 on T4: switch MODEL_ID to the fallback in settings.")
        return (loss, out) if return_outputs else loss


class StopBeforeKaggleKills(TrainerCallback):
    """Kaggle discards a run at 12 h. Stop in time to merge and save, with a checkpoint at this step to resume from."""

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - START > MAX_HOURS * 3600 or free_gb() < MIN_FREE_GB:
            print(f"stopping at update {state.global_step} with a checkpoint: {time.time() - START:.0f} s, {free_gb():.1f} GB host RAM free. Resume with RESUME = True", flush=True)
            control.should_training_stop = control.should_save = True

    def on_log(self, args, state, control, **kwargs):  # a run killed at update 208 left no traceback: peak host RAM growing towards Kaggle's limit would show here
        print(f"update {state.global_step}, peak host RAM {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:.1f} GB, free {free_gb():.1f} GB, peak GB per GPU", [round(torch.cuda.max_memory_allocated(i) / 2**30, 1) for i in range(torch.cuda.device_count())], flush=True)



delta_rule = qwen.torch_chunk_gated_delta_rule


def delta_rule_in_parts(query, key, value, g, beta, initial_state=None, **kwargs):
    """The kernel of the 24 linear attention layers: whole, it keeps about 5 GB per layer for backward on the longest conversation. It already carries its state between calls, so run it in checkpointed parts."""
    kwargs["output_final_state"] = True

    def part(q, k, v, g, beta, state):
        return delta_rule(q, k, v, g=g, beta=beta, initial_state=state, **kwargs)

    outs, state = [], initial_state
    for start in range(0, query.shape[1], DELTA_PART):
        pieces = (x[:, start : start + DELTA_PART] for x in (query, key, value, g, beta))
        out, state = checkpoint(part, *pieces, state, use_reentrant=False, preserve_rng_state=False)
        outs.append(out)
    return torch.cat(outs, 1), state


assert DELTA_PART % 64 == 0
_x = [torch.randn(1, DELTA_PART + 100, 2, 8) for _ in range(3)]
_gates = dict(g=-torch.rand(1, DELTA_PART + 100, 2), beta=torch.rand(1, DELTA_PART + 100, 2), use_qk_l2norm_in_kernel=True)
assert torch.allclose(delta_rule_in_parts(*_x, **_gates)[0], delta_rule(*_x, **_gates)[0], rtol=1e-3, atol=1e-4), "parts differ from the whole kernel"
qwen.torch_chunk_gated_delta_rule = delta_rule_in_parts
sdpa.use_gqa_in_sdpa = lambda *args: False  # a T4 has no grouped-key attention kernel: it would fall back to math, 4 float32 tensors of 16 x T^2, about 2 GB each at 5,900 tokens
torch.backends.cuda.enable_math_sdp(False)  # so a fallback stops at the first step with "No available kernel" and not out of memory

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="auto" if torch.cuda.is_available() else None, attn_implementation="sdpa")
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"))  # adapters stay fp32, so the fp16 GradScaler works
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="ckpt",
    per_device_train_batch_size=1,  # one conversation is several thousand tokens
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    max_steps=SMOKE_STEPS if SMOKE else -1,
    learning_rate=LR,
    optim="adamw_torch",  # not the fused default: it skips an overflowed step only if found_inf == 1, and GradScaler sums devices, so two GPUs overflowing together give 2 and NaN goes into every weight
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=1.0,
    fp16=FP16,
    use_cpu=not torch.cuda.is_available(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    label_names=["labels"],
    logging_steps=10,
    disable_tqdm=True,  # a progress bar per update flooded the kernel's output, which stopped flushing for an hour
    save_strategy="steps",
    save_steps=10 if SMOKE else SAVE_STEPS,  # the smoke proves that saving works
    save_total_limit=1,
    report_to="none",
    seed=SEED,
)
trainer = TargetOnlyLoss(
    model=model,
    args=args,
    train_dataset=Conversations(),
    data_collator=lambda batch: {k: torch.tensor([batch[0][k]]) for k in batch[0]},
    callbacks=[StopBeforeKaggleKills()],
)
try:
    trainer.train(resume_from_checkpoint=CHECKPOINT)
finally:
    print("peak GB per GPU:", [round(torch.cuda.max_memory_allocated(i) / 2**30, 1) for i in range(torch.cuda.device_count())], "GradScaler scale:", trainer.accelerator.scaler.get_scale() if trainer.accelerator.scaler else None)
trainer.save_model("llm_lora")

In [ ]:
%%writefile merge.py
"""Merge the new LoRA into the old checkpoint's weights, on CPU."""
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from settings import *

model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(LLM_OLD, dtype=DTYPE), "llm_lora").merge_and_unload()
params = sum(p.numel() for p in model.parameters())
assert params < 5e9, f"over the LLM limit: {params:,}"
print(f"LLM parameters: {params:,}")
model.save_pretrained("llm_model")
AutoTokenizer.from_pretrained(LLM_OLD).save_pretrained("llm_model")

In [ ]:
%%writefile why_from_traces.py
"""DeepSeek train traces (Kaggle notebook, WHY = True) -> rationales.json for the student SFT notebook (bowen.ipynb, WHY = True).

rationales.json = {row id: [reason per history call..., reason per final call..., reason for done]}; the SFT notebook takes the
last len(labelled) entries. This script copies --base and replaces the final-call reasons of a row with DeepSeek's written
reasons only when the trace is a clean copy of gold: score 1.0, every call executed (no bounced/schema/repeat calls), one call
per step, the same tool names in gold order, and a short non-empty reason before every call. History and done reasons stay.
Sources: train split only (R-RES-3, F-2): gold labels + DeepSeek run on gold text (R-RES-2, development only).

Usage: python student/why_from_traces.py --traces traces_train_deepseek1545_why.jsonl [more traces] --base rationales.json --out rationales_ds.json
       python student/why_from_traces.py --self-check
"""
import argparse
import collections
import json
import re
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
MAX_CHARS = 300  # the student writes the reason before every action and has 30 s per sample (R-RUN-2); longer ones are not cut, the base reason stays


def n_history_calls(x):
    return sum(len(t.get("tool_calls") or []) for t in x["history"] if t["role"] == "assistant")


def reasons(x, trace):
    """DeepSeek's reasons for x's final calls in gold order, or (None, why not)."""
    if not x["label"]["tool_calls"]:
        return None, "gold has no calls"
    if not trace or trace.get("score") != 1.0:
        return None, "score below 1"
    steps = [s for s in trace["steps"] if s["calls"]]
    if any(len(s["calls"]) > 1 for s in steps):
        return None, "several calls in one step"
    if any(s["calls"][0]["status"] != "ok" for s in steps):
        return None, "a call was not executed"
    if [s["calls"][0]["name"] for s in steps] != [c["name"] for c in x["label"]["tool_calls"]]:
        return None, "other call order"
    out = [re.sub(r"\s+", " ", s.get("text") or "").strip() for s in steps]
    if not all(out):
        return None, "no written reason"
    if max(map(len, out)) > MAX_CHARS:
        return None, f"reason over {MAX_CHARS} chars"
    return out, "replaced"


def convert(rows, traces, base):
    out, why = {}, collections.Counter()
    for x in rows:
        b = base[x["id"]]
        n = len(x["label"]["tool_calls"])
        if len(b) != n_history_calls(x) + n + 1:
            raise SystemExit(f"{x['id']}: base has {len(b)} reasons, expected history calls + final calls + done")
        r, w = reasons(x, traces.get(x["id"])) if x["id"] in traces else (None, "no trace")
        why[w] += 1
        out[x["id"]] = b[: len(b) - n - 1] + r + b[-1:] if r else b
    return out, why


def self_check():
    call = lambda name: {"name": name, "arguments": {}}  # noqa: E731
    x = {"id": "t1", "history": [{"role": "user"}, {"role": "assistant", "tool_calls": [call("a.read")]}],
         "label": {"tool_calls": [call("a.read"), call("a.write")]}}
    base = {"t1": ["h", "f1", "f2", "done"]}
    step = lambda name, text, status="ok": {"text": text, "calls": [dict(call(name), status=status)]}  # noqa: E731
    good = {"id": "t1", "score": 1.0, "steps": [step("a.read", " cần  ID "), step("a.write", "ghi"), {"text": "xong", "calls": []}]}
    assert convert([x], {"t1": good}, base)[0]["t1"] == ["h", "cần ID", "ghi", "done"]
    bad = [dict(good, score=0.5), dict(good, steps=[step("a.write", "ghi"), step("a.read", "đọc")]),
           dict(good, steps=[step("a.read", "x", "rare_arg"), step("a.read", "x"), step("a.write", "y")]),
           dict(good, steps=[step("a.read", ""), step("a.write", "y")]),
           dict(good, steps=[{"text": "x", "calls": [dict(call("a.read"), status="ok"), dict(call("a.write"), status="ok")]}])]
    for t in bad:
        assert convert([x], {"t1": t}, base)[0]["t1"] == base["t1"], t
    assert convert([x], {}, base)[1] == {"no trace": 1}
    empty = dict(x, history=[], label={"tool_calls": []})
    assert convert([empty], {"t1": dict(good, steps=[{"text": "ngoài phạm vi", "calls": []}])}, {"t1": ["done"]})[0]["t1"] == ["done"]
    print("self-check ok")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--traces", nargs="+", help="train traces from the notebook's cell 4 with WHY = True")
    ap.add_argument("--base", help="existing rationales.json with every train row (the SFT notebook asserts that)")
    ap.add_argument("--out", help="new rationales.json")
    ap.add_argument("--train", default=str(ROOT / "train+public_test/train/train.jsonl"))
    ap.add_argument("--self-check", action="store_true")
    a = ap.parse_args()
    if a.self_check:
        return self_check()
    if not (a.traces and a.base and a.out):
        ap.error("--traces, --base and --out are required")
    rows = [json.loads(l) for l in open(a.train, encoding="utf-8") if l.strip()]
    traces = {}
    for f in a.traces:
        for l in open(f, encoding="utf-8"):
            o = json.loads(l)
            if not o["id"].startswith("train_"):
                raise SystemExit(f"{f}: {o['id']} is not a train row; only train traces may become training data (R-RES-3)")
            traces[o["id"]] = o
    out, why = convert(rows, traces, json.load(open(a.base, encoding="utf-8")))
    json.dump(out, open(a.out, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"{len(out)} rows -> {a.out}; final-call reasons from DeepSeek in {why['replaced']} rows")
    print("kept the base reasons because:", dict(why))


if __name__ == "__main__":
    main()


In [ ]:
# 3. Inputs, transcripts, row order
from settings import *

shutil.copy(Path(LLM_OLD).parent / "lifemate.py", "lifemate.py")  # saved next to the new weights too
import lifemate as lm

print("checkpoint:", LLM_OLD, "\ntrain text:", NOISY_ASR, "\npublic text:", ASR_PUBLIC, "\nreasons:", WHY_BASE)
rows, public = lm.load_split(ROOT, "train"), lm.load_split(ROOT, "public_test")
by_id = {x["id"]: x for x in rows}
asr = {k: v for k, v in json.load(open(ASR_PUBLIC, encoding="utf-8")).items() if k.startswith("public_test/")}
asr |= {f"train/{k}": v for k, v in json.load(open(NOISY_ASR, encoding="utf-8")).items()}
for split, part in (("public_test", public), ("train", rows)):
    miss = [c for x in part for c in lm.clips(x) if f"{split}/{c}" not in asr]
    assert not miss, f"{len(miss)} {split} clips have no transcript, e.g. {miss[:3]}: is this the full 01c output?"
json.dump(asr, open(ASR_JSON, "w", encoding="utf-8"), ensure_ascii=False)
whys = json.load(open(WHY_BASE, encoding="utf-8"))
assert all(x["id"] in whys for x in rows) and all(k.startswith("train_") for k in whys), "rationales.json must hold the train ids and nothing else (R-RES-3)"

fit, unseen, seen = lm.split_rows(rows)
held = unseen + seen
gate = [x["id"] for x in held[::5]]  # measured before and after, never trained on in this notebook
rest = [x["id"] for x in held if x["id"] not in set(gate)]
fit_ids = [x["id"] for x in fit]
random.Random(SEED).shuffle(fit_ids)
order = (rest + fit_ids) if SMOKE else gate + rest + fit_ids  # smoke: 16 rows that may be trained on, so training runs too
json.dump(order, open("roll_ids.json", "w"))
json.dump(gate, open("gate_ids.json", "w"))
print(f"rollout order: {len(gate)} gate rows, {len(rest)} other held-out rows, {len(fit_ids)} train-split rows; {ROLL_MIN:.0f} min")


def infer(split, tag, llm, minutes, ids=""):
    """Two infer.py processes, one per T4, until done or `minutes` are over. Returns their lines."""
    env = {**os.environ, "TAG": tag, "LLM_DIR": llm, "STOP_AT": str(time.time() + 60 * minutes), "IDS_JSON": ids, "TOKENIZERS_PARALLELISM": "false"}
    ps = [subprocess.Popen([sys.executable, "-u", "infer.py", split, "asr", str(s)], env={**env, "CUDA_VISIBLE_DEVICES": str(s)},
                           stdout=open(f"infer_{tag}_{s}.log", "w", encoding="utf-8"), stderr=subprocess.STDOUT) for s in range(SHARDS)]
    files, t0 = [f"pred_{tag}_{split}_{s}.jsonl" for s in range(SHARDS)], time.time()
    while any(p.poll() is None for p in ps):
        time.sleep(5 if os.environ.get("ZOLA_IN") else 120)
        print(f"  {tag} {split}: {sum(sum(1 for _ in open(f, encoding='utf-8')) for f in files if os.path.exists(f))} samples after {(time.time() - t0) / 60:.0f} min", flush=True)
    for s, p in enumerate(ps):
        if p.returncode:
            print(f"shard {s} exited with {p.returncode}:\n" + open(f"infer_{tag}_{s}.log", encoding="utf-8", errors="replace").read()[-2000:])
    return [json.loads(l) for f in files if os.path.exists(f) for l in open(f, encoding="utf-8")]

In [ ]:
# 4. K2 Horizon on the same train rows, in the background (CPU). Optional: any problem here only prints a note
K2, K2_TRACES, K2_STOP = None, Path("traces_train_k2_why.jsonl").resolve(), Path("K2_STOP").resolve()
if USE_K2:
    try:
        if not os.environ.get("IFM_API_KEY"):
            from kaggle_secrets import UserSecretsClient
            os.environ["IFM_API_KEY"] = UserSecretsClient().get_secret("IFM_API_KEY").strip()
        Z = Path(os.environ.get("ZOLA_ROOT", "/tmp/ZOLA"))  # writable copy in the layout solve.py expects
        solve = next((p for p in Path(IN).rglob("solve.py") if (p.parent / "transcripts_gemini_public_test.json").is_file()), None)
        assert solve and '"--why"' in solve.read_text(encoding="utf-8") and "KEY_AT" in solve.read_text(encoding="utf-8"), "zola_deepseek_bundle with the current solve.py is not attached (unpacked)"
        if not (Z / "api_agent/solve.py").is_file():
            shutil.copytree(solve.parent, Z / "api_agent", ignore=shutil.ignore_patterns("__pycache__"))
            shutil.copytree(ROOT / "train", Z / "train+public_test/train", ignore=shutil.ignore_patterns("__pycache__", "audio"))
        K2_STOP.unlink(missing_ok=True)
        k2_ids = (rest + fit_ids)[: 16 if SMOKE else K2_ROWS]  # train rows only (R-RES-3); the gate rows are not trained on, so they need no reasons
        K2 = subprocess.Popen([sys.executable, "-u", "solve.py", "--split", "train", "--backend", "ifm", "--why", "--ids", ",".join(k2_ids),
                               "--workers", str(K2_WORKERS), "--cache", str(K2_TRACES.with_name("cache_train_k2_why.jsonl")), "--traces", str(K2_TRACES),
                               "--stop-file", str(K2_STOP)], cwd=Z / "api_agent", stdout=open("run_k2.log", "a", encoding="utf-8"),
                              stderr=subprocess.STDOUT, env={**os.environ, "PYTHONIOENCODING": "utf-8"})
        print(f"K2 started on {len(k2_ids)} train rows with {len(os.environ['IFM_API_KEY'].replace(',', ' ').split())} key(s); log: run_k2.log")
    except Exception as e:
        print(f"K2 skipped ({type(e).__name__}: {e}); the reasons of rationales.json are used")

In [ ]:
# 5. Rollouts of the old checkpoint on train rows (reward = official score), then the reward-selected training rows
roll = {x["id"]: x for x in infer("train", "old", LLM_OLD, ROLL_MIN, "roll_ids.json")}
assert roll, "no rollout finished: see infer_old_0.log"
if K2:  # no new rows; the running ones finish and are saved
    K2_STOP.touch()
    try:
        K2.wait(timeout=240)
    except subprocess.TimeoutExpired:
        K2.kill()
    print("K2 log tail:\n" + open("run_k2.log", encoding="utf-8", errors="replace").read()[-800:])
k2 = {}
for l in open(K2_TRACES, encoding="utf-8") if K2_TRACES.is_file() else ():
    try:
        o = json.loads(l)
    except ValueError:  # a line cut by the kill
        continue
    if o["id"].startswith("train_"):
        k2[o["id"]] = o

held_set, gate_set = {x["id"] for x in held}, set(gate)
done = [i for i in order if i in roll and i not in gate_set]
fail = [i for i in done if roll[i]["score"] < 1]
ok_new = [i for i in done if roll[i]["score"] == 1 and i in held_set]  # right, but on rows the checkpoint may never have trained on
ok_old = [i for i in done if roll[i]["score"] == 1 and i not in held_set]
for name, ids in (("gate", gate), ("other held-out", rest), ("train split", fit_ids)):
    r = [roll[i]["score"] for i in ids if i in roll]
    if r:
        print(f"{name:15s} n={len(r):4d} reward {100 * st.mean(r):6.2f}  failed {sum(x < 1 for x in r)}")
sec = st.mean(x["sec"] for x in roll.values())
WHY_ON = sum('"why"' in (x["raw"] or [""])[0] for x in roll.values()) > len(roll) / 2
json.dump({"why": WHY_ON}, open("rl_config.json", "w"))
PUBLIC_MIN = len(public) / SHARDS * sec * 1.15 / 60 + 5  # measured speed of this model on this machine, 15% spare, model load
TRAIN_MIN = 8 if SMOKE else min(TRAIN_CAP, left() - 15 - GATE_MIN - PUBLIC_MIN * PUBLIC_RUN - 5)
rl = (fail * 2 + ok_new + ok_old)[: max(GRAD_ACCUM, int((TRAIN_MIN - 6) * 60 / SEC_PER_CONV))]
json.dump(rl, open("rl_rows.json", "w"))

from why_from_traces import convert
new_whys, why_stats = convert([by_id[i] for i in sorted(set(rl))], k2, whys)
json.dump(whys | new_whys, open(WHY_JSON, "w", encoding="utf-8"), ensure_ascii=False)
k2_ok = sum(o.get("score") == 1.0 for o in k2.values())
print(f"{sec:.1f} s per sample -> a public run takes about {PUBLIC_MIN:.0f} min ({'here' if PUBLIC_RUN else 'in the eval notebook'}), training gets {TRAIN_MIN:.0f} min; {left():.0f} min left")
print(f"checkpoint writes reasons: {WHY_ON}; K2 traces: {len(k2)} rows, {k2_ok} with reward 1; K2 reasons used in {why_stats['replaced']} of {len(set(rl))} training rows")
print(f"training rows: {len(rl)} = {len([i for i in rl if i in set(fail)])} from {len(fail)} failed rollouts (twice) + {len([i for i in rl if i in set(ok_new)])} held-out successes + {len([i for i in rl if i in set(ok_old)])} other successes")

In [ ]:
# 6. Fine-tune the old checkpoint on the selected rows, merge. A failure here does not stop the notebook
NEW = None
if TRAIN_MIN >= (0 if SMOKE else 15) and rl:
    env = {**os.environ, "TRAIN_HOURS": str((TRAIN_MIN - 1) / 60), "LLM_DIR": LLM_OLD, "TOKENIZERS_PARALLELISM": "false", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    rc = run_logged([sys.executable, "-u", "train_llm.py"], "train.log", env)  # exit code 137 means killed, usually out of host RAM
    if rc == 0 and os.path.isdir("llm_lora"):
        rc = run_logged([sys.executable, "-u", "merge.py"], "train.log", env)
    if rc == 0 and os.path.isfile("llm_model/config.json"):
        NEW = "llm_model"
    shutil.rmtree("ckpt", ignore_errors=True)  # only the stop checkpoint, the adapter is in llm_lora
print(f"new model: {NEW}; {left():.0f} min left" if NEW else "NO NEW MODEL (training skipped or failed, see train.log): there is nothing new to evaluate or to put in the Docker")

In [ ]:
# 7. Optional (PUBLIC_RUN): public run with the new model (offline text: 01c transcripts), submission, validation
if PUBLIC_RUN and (NEW or SMOKE):
    pub = infer("public_test", "new", NEW or LLM_OLD, left() - 3)
    assert len({x["id"] for x in pub}) == len(pub)
    assert SMOKE or len(pub) == len(public), f"{len(pub)} lines, expected {len(public)}: see infer_new_*.log"
    with open("submission.jsonl", "w", encoding="utf-8") as f:
        for x in pub:
            f.write(json.dumps({k: x[k] for k in ("id", "tool_calls", "state_hash")}, ensure_ascii=False) + "\n")
    import collections
    print(collections.Counter(x["stop"] for x in pub).most_common(), "|", sum(x["sec"] > 30 for x in pub), "samples over 30 s, slowest", f"{max(x['sec'] for x in pub):.1f} s")
    print(f"{len(pub)} lines, {sum(x['failed'] for x in pub)} failed, {sum(x['invalid'] for x in pub)} stopped on an invalid call, model: {NEW or 'OLD checkpoint'}")
    r = subprocess.run([sys.executable, "-m", "zaic_tools", "--split", "public_test", "--root", "..", "--validate", os.path.abspath("submission.jsonl")],
                       cwd=ROOT / "public_test", capture_output=True, text=True, env={**os.environ, "PYTHONPATH": ".", "PYTHONDONTWRITEBYTECODE": "1"})
    print(r.stdout[-1500:], r.stderr[-1500:])
else:
    print('no public run here: attach this output (llm_model/, lifemate.py) to the eval notebook')

In [ ]:
# 8. Gate: new model against the old model's rollouts on the same held-out rows. The only clean comparison: the other held-out rows were trained on
summary = {"old_model": LLM_OLD, "new_model": NEW, "rollouts": len(roll), "failed_rollouts": len(fail), "training_rows": len(rl),
           "k2_rows": len(k2), "k2_reasons_used": why_stats["replaced"], "reasons": WHY_ON, "train_text": os.path.basename(NOISY_ASR)}
if NEW and (SMOKE or left() > 8):
    new = {x["id"]: x for x in infer("train", "gate", NEW, 20 if SMOKE else min(25, left() - 2), "gate_ids.json")}
    both = [i for i in gate if i in new and i in roll]
    if both:
        o, n = [roll[i]["score"] for i in both], [new[i]["score"] for i in both]
        summary |= {"gate_rows": len(both), "gate_old": 100 * st.mean(o), "gate_new": 100 * st.mean(n),
                    "gate_better": sum(a > b for a, b in zip(n, o)), "gate_worse": sum(a < b for a, b in zip(n, o))}
        print(f"gate, {len(both)} held-out rows: old {summary['gate_old']:.2f} -> new {summary['gate_new']:.2f}, better on {summary['gate_better']}, worse on {summary['gate_worse']}")
        print("With this few rows only a clear gap means something. If the old checkpoint was trained with FINAL = True it has seen these rows, which favours neither model.")
else:
    print("gate skipped:", "no new model" if not NEW else "no time left")
json.dump(summary, open("rl_summary.json", "w"), indent=1)
print(summary, f"\n{left():.0f} min before {DEADLINE_VN}; files: llm_model/, lifemate.py, rl_summary.json" + (", submission.jsonl" if PUBLIC_RUN else ""))